# Automated FST immobility threshold detection and validation

This notebook develops and validates an automated immobility-scoring method from EthoVision XT activity output. The primary workflow uses manually annotated videos to tune strain-specific activity thresholds, evaluates those thresholds by leave-one-video-out validation, and then tests performance on an independent set of manually annotated videos.

**Primary workflow:** manual ground truth + EthoVision activity -> zoom correction -> smoothing -> threshold tuning -> cross-validation -> independent test-set validation -> whole-video agreement and F1 summaries.

The final section contains exploratory thresholding approaches (including hysteresis and HMM trials) that were investigated but are not part of the primary method.


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook

# -----------------------------------------------------------------------------
# Project paths
# -----------------------------------------------------------------------------
# Run the notebook from the repository root, or change PROJECT_ROOT below.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "fst_method"
RESULTS_DIR = PROJECT_ROOT / "results" / "fst_method"
FIGURE_DIR = RESULTS_DIR / "figures"

TRAINING_ACTIVITY_DIR = DATA_DIR / "ethovision_training"
TEST_ACTIVITY_DIR = DATA_DIR / "ethovision_test"
MANUAL_TRAINING_FILE = DATA_DIR / "manual_ground_truth_annotations.csv"
MANUAL_TEST_FILE = DATA_DIR / "ground_truth_test.txt"
FST_RUNS_FILE = DATA_DIR / "FST_runs.xlsx"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# -----------------------------------------------------------------------------
# Load manually annotated training videos
# -----------------------------------------------------------------------------
# Expected format: one video per column and one 5-s scoring interval per row.
# Optional metadata columns (interval/start_time_s/end_time_s) are ignored here.
manual = pd.read_csv(MANUAL_TRAINING_FILE)

metadata_cols = ["interval", "start_time_s", "end_time_s"]
manual = manual.drop(columns=[c for c in metadata_cols if c in manual.columns])

# Accept either explicit labels or the original m/i shorthand.
manual = manual.replace({"mobile": "m", "immobile": "i"})
manual.columns = [int(c) if str(c).isdigit() else c for c in manual.columns]

print(f"Loaded manual annotations for {manual.shape[1]} training videos.")
print(f"Scoring intervals per video: {manual.shape[0]}")


## 1. Read EthoVision activity from training videos

The helper below extracts the experiment identifier from cell B10 and the activity series from column N of each EthoVision workbook.


In [ ]:
def build_automated_df(folder):
    folder = Path(folder)
    data = {}
    skipped = []

    for file in folder.glob("*.xlsx"):
        try:
            wb = load_workbook(file, read_only=True, data_only=True)
            ws = wb[wb.sheetnames[0]]

            # Read experiment name from B10
            exp_name = ws["B10"].value
            if exp_name is None:
                skipped.append((file.name, "B10 empty"))
                wb.close()
                continue

            exp_name = str(exp_name).strip()
            exp_name = re.sub(r"^Exp\s*", "", exp_name)   # remove leading 'Exp'

            # Read N37:N9037 inclusive
            values = [row[0] for row in ws.iter_rows(min_row=37, max_row=9037,
                                                     min_col=14, max_col=14,
                                                     values_only=True)]

            wb.close()

            data[exp_name] = values
            print(f"Loaded {file.name} -> {exp_name}")

        except Exception as e:
            skipped.append((file.name, str(e)))
            print(f"Skipped {file.name}: {e}")

    automated_df = pd.DataFrame(data)
    return automated_df, skipped

In [ ]:
automated_df, skipped = build_automated_df(TRAINING_ACTIVITY_DIR)


## 2. Training-set metadata and zoom correction

Map videos to strain groups and correct activity values for differences in recording zoom before threshold tuning.


In [ ]:
strain_by_exp = {
    2: "MRL/MpJ",
    135: "MRL/MpJ",
    148: "MRL/MpJ",
    184: "MRL/MpJ",

    299: "MRL/MpJ_2",
    413: "MRL/MpJ_2",
    414: "MRL/MpJ_2",
    419: "MRL/MpJ_2",
    449: "MRL/MpJ_2",
    
    68: "DBA/2J",
    69: "DBA/2J",
    122: "DBA/2J",
    191: "DBA/2J",
    288: "DBA/2J",
    311: "DBA/2J",
    344: "DBA/2J",

    60: "CBA/J",
    66: "CBA/J",
    171: "CBA/J",
    167: "CBA/J",
    168: "CBA/J",
    169: "CBA/J",
    380: "CBA/J",
    226: "CBA/J",
    200: "CBA/J",
    219: "CBA/J",
    
    72: "C57BL/6J",
    73: "C57BL/6J",
    307: "C57BL/6J",
    360: "C57BL/6J",
    281: "C57BL/6J",
    255: "C57BL/6J",
    209: "C57BL/6J"
}

In [ ]:
df2 = pd.read_excel(FST_RUNS_FILE)

if "Experiment" not in df2.columns or "Pixel Distance" not in df2.columns:
    raise ValueError("The Excel file must contain 'Experiment' and 'Pixel Distance' columns.")

# Use the first recorded pixel distance as the reference scale.
reference_distance = df2["Pixel Distance"].iloc[0]

# Compute relative pixel distance and squared zoom correction factor.
df2["Relative Pixel Distance"] = df2["Pixel Distance"] / reference_distance
df2["Zoom"] = df2["Relative Pixel Distance"] ** 2

df2[["Experiment", "Pixel Distance", "Relative Pixel Distance", "Zoom"]]


In [ ]:

zoom_map = dict(zip(df2["Experiment"].astype(str), df2["Zoom"]))

ethovision_df = automated_df.copy()
ethovision_df.columns = ethovision_df.columns.astype(str)

missing_zoom = []

for col in ethovision_df.columns:
    if col in zoom_map:
        ethovision_df[col] = pd.to_numeric(ethovision_df[col], errors="coerce") / zoom_map[col]
    else:
        missing_zoom.append(col)

print("Columns missing zoom correction:", missing_zoom)

ethovision_df.columns = [int(c) if str(c).isdigit() else c for c in ethovision_df.columns]

In [ ]:
ethovision_df = ethovision_df.iloc[:-1]

In [ ]:
manual.columns = [int(c) for c in manual.columns]
ethovision_df.columns = [int(c) for c in ethovision_df.columns]

## 3. Smooth activity traces

Apply the same rolling-window smoothing used for threshold detection before comparing automated activity with manual annotations.


In [ ]:

ethovision_smooth = ethovision_df.copy()

window = 100 # 4 seconds

for col in ethovision_smooth.columns:
    ethovision_smooth[col] = (
        pd.to_numeric(ethovision_smooth[col], errors="coerce")
        .rolling(window=window, center=True, min_periods=1)
        .mean()
    )


## 4. Tune strain-specific activity thresholds

Search candidate thresholds against the manually annotated training videos and summarize immobility F1 performance by strain.


In [ ]:
strain_order = ["MRL/MpJ", "MRL/MpJ_2", "DBA/2J", "CBA/J", "C57BL/6J"]

f1_mat, thresholds, exp_to_strain, best_thresholds_by_strain, strain_order = build_combined_f1_heatmap(
    manual_df=manual,
    ethovision_df=ethovision_smooth,   # your 3s-smoothed df
    strain_by_exp=strain_by_exp,
    strain_order=strain_order,
    n_thresholds=100
)

plot_combined_f1_heatmap(
    f1_mat,
    exp_to_strain,
    best_thresholds_by_strain,
    strain_order
)


from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
)
import numpy as np
import pandas as pd

def print_best_threshold_metrics_both_classes(
    manual_df,
    ethovision_df,
    best_thresholds_by_strain,
    strain_by_exp,
    strain_order,
):
    print(
    f"{'Strain':<15} {'Threshold':>10} "
    f"{'F1_i':>8} "
)
    print("-" * 120)

    rows = []

    for strain in strain_order:
        t_best = best_thresholds_by_strain.get(strain)
        if t_best is None:
            continue

        strain_exps = [
            exp for exp, s in strain_by_exp.items()
            if s == strain and exp in manual_df.columns and exp in ethovision_df.columns
        ]

        y_true_list, y_pred_list = [], []

        for exp in strain_exps:
            manual_series = manual_df[exp]
            activity_series = pd.to_numeric(ethovision_df[exp], errors="coerce")

            frame_idx, y_true = get_ground_truth_frames_aligned(
                manual_series,
                activity_len=len(activity_series),
                fps=FPS,
                interval_s=WINDOW_S,
                trust_s=1,
                align="center"
            )

            valid = frame_idx < len(activity_series)
            frame_idx, y_true = frame_idx[valid], y_true[valid]

            x = activity_series.iloc[frame_idx].to_numpy()
            keep = ~np.isnan(x)

            x = x[keep]
            y_true = y_true[keep]

            # y_true: 0 = immobile, 1 = mobile
            # y_pred: 0 = immobile, 1 = mobile
            y_pred = (x > t_best).astype(int)

            y_true_list.append(y_true)
            y_pred_list.append(y_pred)

        y_true_all = np.concatenate(y_true_list)
        y_pred_all = np.concatenate(y_pred_list)

        macro_f1 = f1_score(y_true_all, y_pred_all, average="macro", zero_division=0)
        bal_acc = balanced_accuracy_score(y_true_all, y_pred_all)

        f1_i = f1_score(y_true_all, y_pred_all, pos_label=0, zero_division=0)
        pr_i = precision_score(y_true_all, y_pred_all, pos_label=0, zero_division=0)
        rc_i = recall_score(y_true_all, y_pred_all, pos_label=0, zero_division=0)

        f1_m = f1_score(y_true_all, y_pred_all, pos_label=1, zero_division=0)
        pr_m = precision_score(y_true_all, y_pred_all, pos_label=1, zero_division=0)
        rc_m = recall_score(y_true_all, y_pred_all, pos_label=1, zero_division=0)

        cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

        # Matrix layout:
        #                 Pred immobile   Pred mobile
        # True immobile        cm[0,0]       cm[0,1]
        # True mobile          cm[1,0]       cm[1,1]

        true_i_pred_i = cm[0, 0]
        true_i_pred_m = cm[0, 1]  # false negative for immobility
        true_m_pred_i = cm[1, 0]  # false positive for immobility
        true_m_pred_m = cm[1, 1]

        fp_i = true_m_pred_i
        fn_i = true_i_pred_m

        print(
            f"{strain:<15} {t_best:>10.4f} "
            f"{f1_i:>8.4f} ")

        rows.append({
            "strain": strain,
            
            "threshold": t_best,
            
            "f1": f1_i,
        })

    return pd.DataFrame(rows)

metrics_df = print_best_threshold_metrics_both_classes(
    manual,
    ethovision_smooth,
    best_thresholds_by_strain,
    strain_by_exp,
    strain_order,
)

display(metrics_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

FPS = 25
WINDOW_S = 5
TRUST_S = 2

def get_ground_truth_frames_aligned(
    manual_series,
    activity_len,
    fps=25,
    interval_s=5,
    trust_s=1,
    align="center",
):
    trusted_frames = int(round(trust_s * fps))

    frame_indices = []
    labels = []

    for i, val in enumerate(manual_series):
        val = str(val).strip().lower()

        if val not in ["m", "i"]:
            continue

        label = 1 if val == "m" else 0
        obs_frame = int(round((i + 1) * interval_s * fps))

        if align == "center":
            half = trusted_frames // 2
            start = obs_frame - half
            end = start + trusted_frames
        elif align == "after":
            start = obs_frame
            end = obs_frame + trusted_frames
        elif align == "before":
            start = obs_frame - trusted_frames
            end = obs_frame
        else:
            raise ValueError("align must be 'center', 'after', or 'before'")

        # Skip incomplete windows
        if start < 0 or end > activity_len:
            continue

        inds = np.arange(start, end)

        frame_indices.extend(inds)
        labels.extend([label] * len(inds))

    return np.array(frame_indices), np.array(labels)

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
)
import numpy as np
import pandas as pd


# ============================================================
# Define your fixed thresholds here
# ============================================================

fixed_thresholds_by_strain = {
    "MRL/MpJ": 2.28,       # replace with your chosen threshold
    "MRL/MpJ_2": 2.7213,     # replace or remove if not needed
    "DBA/2J": 0.5884,
    "CBA/J": 1.0297,
    "C57BL/6J": 1.3239,
}

 
def build_combined_f1_heatmap(manual_df, ethovision_df, strain_by_exp, strain_order=None, n_thresholds=80):
    if strain_order is None:
        strain_order = sorted(set(strain_by_exp.values()))

    # experiments ordered by strain blocks
    ordered_exps = []
    exp_to_strain = {}
    for strain in strain_order:
        exps = sorted([
            exp for exp, s in strain_by_exp.items()
            if s == strain and exp in manual_df.columns and exp in ethovision_df.columns
        ])
        ordered_exps.extend(exps)
        for exp in exps:
            exp_to_strain[exp] = strain

    # one global threshold grid across all included experiments
    pooled = []
    for exp in ordered_exps:
        vals = pd.to_numeric(ethovision_df[exp], errors="coerce").dropna().to_numpy()
        if len(vals):
            pooled.append(vals)

    pooled = np.concatenate(pooled)
    thresholds = np.linspace(pooled.min(), pooled.max(), n_thresholds)

    f1_mat = pd.DataFrame(index=ordered_exps, columns=thresholds, dtype=float)

    # fill matrix
    for exp in ordered_exps:
        manual_series = manual_df[exp]
        activity_series = pd.to_numeric(ethovision_df[exp], errors="coerce")
    
        frame_idx, y_true = get_ground_truth_frames_aligned(
            manual_series,
            activity_len=len(activity_series),
            fps=FPS,
            interval_s=WINDOW_S,
            trust_s=TRUST_S,
            align="center",
        )
    
        x = activity_series.iloc[frame_idx].to_numpy()
        keep = ~np.isnan(x)
    
        x = x[keep]
        y_true = y_true[keep]
    
        for t in thresholds:
            y_pred = (x > t).astype(int)  # 1 = mobile, 0 = immobile
            f1_mat.loc[exp, t] = f1_score(
                y_true,
                y_pred,
                pos_label=0,
                zero_division=0,
            )
    # pooled best threshold per strain
    best_thresholds_by_strain = {}
    for strain in strain_order:
        strain_exps = [exp for exp in ordered_exps if exp_to_strain[exp] == strain]
        if not strain_exps:
            continue

        best_t = None
        best_f1 = -1

        for t in thresholds:
            y_true_all = []
            y_pred_all = []

            for exp in strain_exps:
                manual_series = manual_df[exp]
                activity_series = pd.to_numeric(ethovision_df[exp], errors="coerce")

                frame_idx, y_true = get_ground_truth_frames_aligned(manual_series, activity_len=len(activity_series), fps=FPS, interval_s=WINDOW_S, trust_s=TRUST_S, align="center")

                x = activity_series.iloc[frame_idx].to_numpy()
                keep = ~np.isnan(x)
                x = x[keep]
                y_true = y_true[keep]

                y_pred = (x > t).astype(int)

                y_true_all.append(y_true)
                y_pred_all.append(y_pred)

            y_true_all = np.concatenate(y_true_all)
            y_pred_all = np.concatenate(y_pred_all)
            f1 = f1_score(y_true_all, y_pred_all, pos_label=0, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds_by_strain[strain] = best_t

    return f1_mat, thresholds, exp_to_strain, best_thresholds_by_strain, strain_order


def plot_combined_f1_heatmap(f1_mat, exp_to_strain, best_thresholds_by_strain, strain_order):
    fig, ax = plt.subplots(figsize=(13, max(6, 0.42 * len(f1_mat.index))))

    im = ax.imshow(
        f1_mat.values,
        aspect="auto",
        interpolation="nearest"
    )

    # x ticks
    n_cols = f1_mat.shape[1]
    xticks = np.linspace(0, n_cols - 1, min(8, n_cols)).astype(int)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{f1_mat.columns[i]:.2f}" for i in xticks], rotation=45, ha="right")

    # y ticks
    ax.set_yticks(np.arange(len(f1_mat.index)))
    ax.set_yticklabels([str(exp) for exp in f1_mat.index])

    # mark best threshold per experiment
    for r in range(f1_mat.shape[0]):
        c = np.nanargmax(f1_mat.values[r])
        ax.scatter(c, r, marker="o", s=28)

    # strain block separators and labels
    strain_boundaries = []
    row_start = 0

    for strain in strain_order:
        rows = [i for i, exp in enumerate(f1_mat.index) if exp_to_strain[exp] == strain]
        if not rows:
            continue

        first_row = min(rows)
        last_row = max(rows)

        # separator line after each block except last
        strain_boundaries.append(last_row + 0.5)

        # label on left margin
        y_mid = (first_row + last_row) / 2
        ax.text(
            -0.08, y_mid, strain,
            va="center", ha="right",
            transform=ax.get_yaxis_transform(),
            fontsize=10, fontweight="bold"
        )

        # pooled best threshold per strain as vertical dashed line over the block
        t_best = best_thresholds_by_strain[strain]
        c_best = np.argmin(np.abs(f1_mat.columns.to_numpy(dtype=float) - t_best))
        ax.vlines(
            c_best, first_row - 0.5, last_row + 0.5,
            linestyles="dashed", linewidth=1.5
        )

    for y in strain_boundaries[:-1]:
        ax.hlines(y, -0.5, f1_mat.shape[1] - 0.5, linewidth=2)

    ax.set_xlabel("Threshold")
    ax.set_ylabel("Experiment")
    ax.set_title("F1 heatmap across thresholds and experiments, grouped by strain")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("F1 score")

    plt.tight_layout()
    #plt.savefig('threshold_M_I_bystrain.svg', dpi=600)
    #plt.close()
    plt.show()

## 5. Leave-one-video-out validation

Evaluate the selected thresholding approach while holding out one manually annotated video at a time.


In [ ]:
def get_xy_for_experiment_fixed_threshold_test(
    exp,
    manual_df,
    ethovision_df,
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
):
    """
    Extracts trusted-window EthoVision activity values and manual labels
    for one experiment.

    y coding:
        0 = immobile
        1 = mobile
    """
    manual_series = manual_df[exp]
    activity_series = pd.to_numeric(ethovision_df[exp], errors="coerce")

    frame_idx, y_true = get_ground_truth_frames_aligned(
        manual_series,
        activity_len=len(activity_series),
        fps=fps,
        interval_s=interval_s,
        trust_s=trust_s,
        align=align,
    )

    valid = frame_idx < len(activity_series)
    frame_idx = frame_idx[valid]
    y_true = y_true[valid]

    x = activity_series.iloc[frame_idx].to_numpy()
    keep = ~np.isnan(x)

    return x[keep], y_true[keep]


def test_fixed_thresholds_leave_one_video_out(
    manual_df,
    ethovision_df,
    strain_by_exp,
    fixed_thresholds_by_strain,
    strain_order=None,
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
):
    """
    Leave-one-video-out style evaluation using FIXED thresholds.

    Important:
    This does not optimize thresholds.
    It simply tests each video independently using the predefined
    threshold for that video's strain.

    Returns:
        loo_fixed_results: one row per tested video
        loo_fixed_summary: average performance per strain
    """

    manual_df = manual_df.copy()
    ethovision_df = ethovision_df.copy()

    manual_df.columns = [
        int(c) if str(c).isdigit() else c
        for c in manual_df.columns
    ]

    ethovision_df.columns = [
        int(c) if str(c).isdigit() else c
        for c in ethovision_df.columns
    ]

    if strain_order is None:
        strain_order = sorted(set(strain_by_exp.values()))

    rows = []

    for strain in strain_order:
        if strain not in fixed_thresholds_by_strain:
            print(f"Skipping {strain}: no fixed threshold provided")
            continue

        threshold = fixed_thresholds_by_strain[strain]

        strain_exps = sorted([
            exp for exp, s in strain_by_exp.items()
            if (
                s == strain
                and exp in manual_df.columns
                and exp in ethovision_df.columns
            )
        ])

        if len(strain_exps) == 0:
            print(f"Skipping {strain}: no matching videos found")
            continue

        for test_exp in strain_exps:
            x_test, y_test = get_xy_for_experiment_fixed_threshold_test(
                test_exp,
                manual_df,
                ethovision_df,
                fps=fps,
                interval_s=interval_s,
                trust_s=trust_s,
                align=align,
            )

            if len(x_test) == 0:
                continue

            # Same convention as your original analysis:
            # activity > threshold = mobile
            # activity <= threshold = immobile
            y_pred = (x_test > threshold).astype(int)

            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

            rows.append({
                "strain": strain,
                "test_exp": test_exp,
                "threshold": threshold,
                "n_test_frames": len(y_test),

                "immobile_f1": f1_score(
                    y_test, y_pred,
                    pos_label=0,
                    zero_division=0,
                ),
                "immobile_precision": precision_score(
                    y_test, y_pred,
                    pos_label=0,
                    zero_division=0,
                ),
                "immobile_recall": recall_score(
                    y_test, y_pred,
                    pos_label=0,
                    zero_division=0,
                ),

                "mobile_f1": f1_score(
                    y_test, y_pred,
                    pos_label=1,
                    zero_division=0,
                ),
                "mobile_precision": precision_score(
                    y_test, y_pred,
                    pos_label=1,
                    zero_division=0,
                ),
                "mobile_recall": recall_score(
                    y_test, y_pred,
                    pos_label=1,
                    zero_division=0,
                ),

                "macro_f1": f1_score(
                    y_test, y_pred,
                    average="macro",
                    zero_division=0,
                ),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),

                "true_immobile_rate": np.mean(y_test == 0),
                "pred_immobile_rate": np.mean(y_pred == 0),

                "true_immobile_pred_immobile": cm[0, 0],
                "true_immobile_pred_mobile": cm[0, 1],
                "true_mobile_pred_immobile": cm[1, 0],
                "true_mobile_pred_mobile": cm[1, 1],
            })

    loo_fixed_results = pd.DataFrame(rows)

    loo_fixed_summary = (
        loo_fixed_results
        .groupby("strain")
        .agg(
            n_test_videos=("test_exp", "nunique"),

            threshold=("threshold", "first"),

            mean_immobile_f1=("immobile_f1", "mean"),
            sem_immobile_f1=("immobile_f1", lambda x: x.sem()),

            mean_immobile_precision=("immobile_precision", "mean"),
            sem_immobile_precision=("immobile_precision", lambda x: x.sem()),

            mean_immobile_recall=("immobile_recall", "mean"),
            sem_immobile_recall=("immobile_recall", lambda x: x.sem()),

            mean_mobile_f1=("mobile_f1", "mean"),
            sem_mobile_f1=("mobile_f1", lambda x: x.sem()),

            mean_macro_f1=("macro_f1", "mean"),
            sem_macro_f1=("macro_f1", lambda x: x.sem()),

            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            sem_balanced_accuracy=("balanced_accuracy", lambda x: x.sem()),

            mean_accuracy=("accuracy", "mean"),
            sem_accuracy=("accuracy", lambda x: x.sem()),

            mean_true_immobile_rate=("true_immobile_rate", "mean"),
            mean_pred_immobile_rate=("pred_immobile_rate", "mean"),
        )
        .reset_index()
    )

    return loo_fixed_results, loo_fixed_summary

In [ ]:
strain_order = ["MRL/MpJ", "MRL/MpJ_2", "DBA/2J", "CBA/J", "C57BL/6J"]

loo_fixed_results, loo_fixed_summary = test_fixed_thresholds_leave_one_video_out(
    manual_df=manual,
    ethovision_df=ethovision_smooth,
    strain_by_exp=strain_by_exp,
    fixed_thresholds_by_strain=fixed_thresholds_by_strain,
    strain_order=strain_order,
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
)

display(loo_fixed_results.sort_values(["strain", "test_exp"]))
display(loo_fixed_summary)

In [ ]:
plot_order = ["MRL/MpJ", "DBA/2J", "CBA/J", "C57BL/6J"]

plot_df = (
    loo_fixed_summary[loo_fixed_summary["strain"].isin(plot_order)]
    .copy()
)

plot_df["strain"] = pd.Categorical(
    plot_df["strain"],
    categories=plot_order,
    ordered=True,
)
plot_df = plot_df.sort_values("strain")

colors = ["#B8D8D8", "#F6C6B4", "#CDB4DB", "#BDE0A8"]

fig, ax = plt.subplots(figsize=(6.4, 4.2))

x = np.arange(len(plot_df)) * 0.65
x_labs = ["MRL/MpJ (9)", "DBA/2J (7)", "CBA/J (10)", "C57BL/6J (7)"]
bars = ax.bar(
    x,
    plot_df["mean_immobile_f1"],
    yerr=plot_df["sem_immobile_f1"],
    capsize=3,
    color=colors[:len(plot_df)],
    width=0.42,
    edgecolor="none",
    alpha=0.95,
)


ax.set_xticks(x)
ax.set_xticklabels(x_labs)

ax.set_ylabel("Immobility F1", fontsize=12)
ax.set_ylim(0, 1.08)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#CCCCCC")
ax.spines["bottom"].set_color("#CCCCCC")

#ax.grid(axis="y", alpha=0.18)
ax.tick_params(axis="x", labelsize=11)
ax.tick_params(axis="y", labelsize=10)
#plt.title('Accuracy of FST method in the training set', fontdict=None, loc='center', pad=None)
plt.tight_layout()
# plt.savefig(FIGURE_DIR / "calibration_validation.svg", format="svg", dpi=600)
# plt.savefig(FIGURE_DIR / "calibration_validation.pdf", format="pdf", dpi=600)
#plt.close()
plt.show()

## 6. Independent test-set validation

Load an independent set of manual annotations and EthoVision activity files, apply the fixed thresholds learned from the training set, and quantify performance.


In [ ]:
## Parser for unseen files:

import re
import numpy as np
import pandas as pd

def parse_unseen_gt_file(path):
    """
    Parses unseen ground-truth file like:

    135
    mmmmiiii...

    or

    Exp135
    mmmmiiii...

    Returns a dataframe:
        columns = experiment IDs as int
        rows = 5 s manual calls
    """
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    entries = {}

    pattern = re.compile(
        r"^\s*(?:Exp)?(\d+)\s*$\s*^\s*([miMI\s]+)\s*$",
        re.MULTILINE
    )

    for exp_num, seq in pattern.findall(raw):
        labels = re.findall(r"[mi]", seq.lower())
        if labels:
            entries[int(exp_num)] = labels

    if not entries:
        raise ValueError("No experiment blocks found. Check file format.")

    unseen_manual = pd.DataFrame({
        exp: pd.Series(labels)
        for exp, labels in entries.items()
    })

    return unseen_manual

In [ ]:
UNSEEN_GT_PATH = MANUAL_TEST_FILE

unseen_manual = parse_unseen_gt_file(UNSEEN_GT_PATH)
print(f"Loaded manual annotations for {unseen_manual.shape[1]} unseen videos.")


In [ ]:
unseen_ethovision, skipped = build_automated_df(TEST_ACTIVITY_DIR)


In [ ]:
zoom_map = dict(zip(df2["Experiment"].astype(str), df2["Zoom"]))

ethovision_df = unseen_ethovision.copy()
ethovision_df.columns = ethovision_df.columns.astype(str)

missing_zoom = []

for col in ethovision_df.columns:
    if col in zoom_map:
        ethovision_df[col] = pd.to_numeric(ethovision_df[col], errors="coerce") / zoom_map[col]
    else:
        missing_zoom.append(col)

print("Columns missing zoom correction:", missing_zoom)

ethovision_df.columns = [int(c) if str(c).isdigit() else c for c in ethovision_df.columns]

In [ ]:
unseen_ethovision_smooth = ethovision_df.copy()

window = 100 # 4 seconds

for col in unseen_ethovision_smooth.columns:
    unseen_ethovision_smooth[col] = (
        pd.to_numeric(unseen_ethovision_smooth[col], errors="coerce")
        .rolling(window=window, center=True, min_periods=1)
        .mean()
    )


In [ ]:
unseen_strain_by_exp = {
    12: "MRL/MpJ",
    136: "MRL/MpJ",
    137: "MRL/MpJ",
    411: "MRL/MpJ_2",
    423: "MRL/MpJ_2",
    424 : "MRL/MpJ_2",
    267: "DBA/2J",
    6: "DBA/2J",
    57: "DBA/2J",
    76: "CBA/J",
    358: "CBA/J",
    464: "CBA/J",
    351: "C57BL/6J",
    33: "C57BL/6J",
    245: "C57BL/6J"
}

In [ ]:
fixed_thresholds_by_strain = {
    "MRL/MpJ": 2.28,
    "MRL/MpJ_2": 2.7213,
    "DBA/2J": 0.5884,
    "CBA/J": 1.0297,
    "C57BL/6J": 1.3239,
}


In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
)
FPS=25
def get_xy_for_unseen_experiment(
    exp,
    manual_df,
    ethovision_df,
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
):
    """
    Extracts trusted-window EthoVision activity values and manual labels.

    y coding:
        0 = immobile
        1 = mobile
    """
    manual_series = manual_df[exp]
    activity_series = pd.to_numeric(ethovision_df[exp], errors="coerce")

    frame_idx, y_true = get_ground_truth_frames_aligned(
        manual_series,
        activity_len=len(activity_series),
        fps=fps,
        interval_s=interval_s,
        trust_s=trust_s,
        align=align,
    )

    valid = frame_idx < len(activity_series)
    frame_idx = frame_idx[valid]
    y_true = y_true[valid]

    x = activity_series.iloc[frame_idx].to_numpy()
    keep = ~np.isnan(x)

    return x[keep], y_true[keep]


def evaluate_fixed_thresholds_on_unseen_gt(
    unseen_manual_df,
    unseen_ethovision_df,
    unseen_strain_by_exp,
    fixed_thresholds_by_strain,
    strain_order=None,
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
):
    """
    Applies previously computed fixed thresholds to a new unseen GT dataset.

    No threshold optimization is performed.

    Returns:
        unseen_results: one row per experiment/video
        unseen_summary: summary by strain
        unseen_overall: pooled summary across all unseen videos
    """

    manual_df = unseen_manual_df.copy()
    ethovision_df = unseen_ethovision_df.copy()

    manual_df.columns = [
        int(c) if str(c).isdigit() else c
        for c in manual_df.columns
    ]

    ethovision_df.columns = [
        int(c) if str(c).isdigit() else c
        for c in ethovision_df.columns
    ]

    if strain_order is None:
        strain_order = sorted(set(unseen_strain_by_exp.values()))

    rows = []

    for exp, strain in unseen_strain_by_exp.items():

        if exp not in manual_df.columns:
            print(f"Skipping Exp{exp}: not found in unseen manual GT")
            continue

        if exp not in ethovision_df.columns:
            print(f"Skipping Exp{exp}: not found in unseen EthoVision dataframe")
            continue

        if strain not in fixed_thresholds_by_strain:
            print(f"Skipping Exp{exp}: no fixed threshold for strain {strain}")
            continue

        threshold = fixed_thresholds_by_strain[strain]

        x, y_true = get_xy_for_unseen_experiment(
            exp=exp,
            manual_df=manual_df,
            ethovision_df=ethovision_df,
            fps=fps,
            interval_s=interval_s,
            trust_s=trust_s,
            align=align,
        )

        if len(x) == 0:
            print(f"Skipping Exp{exp}: no valid aligned frames")
            continue

        # Same convention as your previous analysis:
        # activity > threshold = mobile
        # activity <= threshold = immobile
        y_pred = (x > threshold).astype(int)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

        rows.append({
            "experiment": exp,
            "strain": strain,
            "threshold": threshold,
            "n_frames": len(y_true),

            "immobile_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
            "immobile_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
            "immobile_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),

            "mobile_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            "mobile_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            "mobile_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),

            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "accuracy": accuracy_score(y_true, y_pred),

            "true_immobile_rate": np.mean(y_true == 0),
            "pred_immobile_rate": np.mean(y_pred == 0),

            "true_immobile_pred_immobile": cm[0, 0],
            "true_immobile_pred_mobile": cm[0, 1],
            "true_mobile_pred_immobile": cm[1, 0],
            "true_mobile_pred_mobile": cm[1, 1],
        })

    unseen_results = pd.DataFrame(rows)

    if unseen_results.empty:
        raise ValueError("No valid unseen experiments were evaluated.")

    unseen_summary = (
        unseen_results
        .groupby("strain")
        .agg(
            n_videos=("experiment", "nunique"),
            n_frames=("n_frames", "sum"),

            threshold=("threshold", "first"),

            mean_immobile_f1=("immobile_f1", "mean"),
            sem_immobile_f1=("immobile_f1", lambda x: x.sem()),

            mean_immobile_precision=("immobile_precision", "mean"),
            sem_immobile_precision=("immobile_precision", lambda x: x.sem()),

            mean_immobile_recall=("immobile_recall", "mean"),
            sem_immobile_recall=("immobile_recall", lambda x: x.sem()),

            mean_mobile_f1=("mobile_f1", "mean"),
            sem_mobile_f1=("mobile_f1", lambda x: x.sem()),

            mean_macro_f1=("macro_f1", "mean"),
            sem_macro_f1=("macro_f1", lambda x: x.sem()),

            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            sem_balanced_accuracy=("balanced_accuracy", lambda x: x.sem()),

            mean_accuracy=("accuracy", "mean"),
            sem_accuracy=("accuracy", lambda x: x.sem()),

            mean_true_immobile_rate=("true_immobile_rate", "mean"),
            mean_pred_immobile_rate=("pred_immobile_rate", "mean"),
        )
        .reset_index()
    )

    # Pooled across all unseen frames
    pooled_y_true = []
    pooled_y_pred = []

    for _, row in unseen_results.iterrows():
        exp = row["experiment"]
        strain = row["strain"]
        threshold = row["threshold"]

        x, y_true = get_xy_for_unseen_experiment(
            exp=exp,
            manual_df=manual_df,
            ethovision_df=ethovision_df,
            fps=fps,
            interval_s=interval_s,
            trust_s=trust_s,
            align=align,
        )

        y_pred = (x > threshold).astype(int)

        pooled_y_true.append(y_true)
        pooled_y_pred.append(y_pred)

    pooled_y_true = np.concatenate(pooled_y_true)
    pooled_y_pred = np.concatenate(pooled_y_pred)

    pooled_cm = confusion_matrix(pooled_y_true, pooled_y_pred, labels=[0, 1])

    unseen_overall = pd.DataFrame([{
        "n_videos": unseen_results["experiment"].nunique(),
        "n_frames": len(pooled_y_true),

        "immobile_f1": f1_score(pooled_y_true, pooled_y_pred, pos_label=0, zero_division=0),
        "immobile_precision": precision_score(pooled_y_true, pooled_y_pred, pos_label=0, zero_division=0),
        "immobile_recall": recall_score(pooled_y_true, pooled_y_pred, pos_label=0, zero_division=0),

        "mobile_f1": f1_score(pooled_y_true, pooled_y_pred, pos_label=1, zero_division=0),
        "mobile_precision": precision_score(pooled_y_true, pooled_y_pred, pos_label=1, zero_division=0),
        "mobile_recall": recall_score(pooled_y_true, pooled_y_pred, pos_label=1, zero_division=0),

        "macro_f1": f1_score(pooled_y_true, pooled_y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(pooled_y_true, pooled_y_pred),
        "accuracy": accuracy_score(pooled_y_true, pooled_y_pred),

        "true_immobile_rate": np.mean(pooled_y_true == 0),
        "pred_immobile_rate": np.mean(pooled_y_pred == 0),

        "true_immobile_pred_immobile": pooled_cm[0, 0],
        "true_immobile_pred_mobile": pooled_cm[0, 1],
        "true_mobile_pred_immobile": pooled_cm[1, 0],
        "true_mobile_pred_mobile": pooled_cm[1, 1],
    }])

    return unseen_results, unseen_summary, unseen_overall

In [ ]:
strain_order = ["MRL/MpJ", "DBA/2J", "CBA/J", "C57BL/6J"]

unseen_results, unseen_summary, unseen_overall = evaluate_fixed_thresholds_on_unseen_gt(
    unseen_manual_df=unseen_manual,
    unseen_ethovision_df=unseen_ethovision_smooth,
    unseen_strain_by_exp=unseen_strain_by_exp,
    fixed_thresholds_by_strain=fixed_thresholds_by_strain,
    strain_order=strain_order,
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
)

display(unseen_results.sort_values(["strain", "experiment"]))
display(unseen_summary)
display(unseen_overall)

In [ ]:
plot_order = ["MRL/MpJ", "DBA/2J", "CBA/J", "C57BL/6J"]

plot_df = unseen_summary[unseen_summary["strain"].isin(plot_order)].copy()

plot_df["strain"] = pd.Categorical(
    plot_df["strain"],
    categories=plot_order,
    ordered=True,
)
plot_df = plot_df.sort_values("strain")

colors = ["#B8D8D8", "#F6C6B4", "#CDB4DB", "#BDE0A8"]

fig, ax = plt.subplots(figsize=(6.4, 4.2))

x = np.arange(len(plot_df)) * 0.65

bars = ax.bar(
    x,
    plot_df["mean_immobile_f1"],
    yerr=plot_df["sem_immobile_f1"],
    capsize=3,
    color=colors[:len(plot_df)],
    width=0.42,
    edgecolor="none",
    alpha=0.95,
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["strain"])

ax.set_ylabel("F1 score", fontsize=16)
ax.set_ylim(0, 1.08)
ax.set_yticks(np.arange(0, 1.01, 0.1))

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#CCCCCC")
ax.spines["bottom"].set_color("#CCCCCC")

ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
"""
plt.savefig(
     FIGURE_DIR / "gt_validation.svg",
     format="svg",
     dpi=600,
 )
plt.savefig(
    FIGURE_DIR / "gt_validation.pdf",
     format="pdf",
     dpi=600,
 )
plt.close()
"""
plt.show()

In [ ]:
unseen_summary

## 7. Whole-video agreement

Convert manual and automated calls to total immobility time per video and compare them using correlation/regression.


In [ ]:
manual_corr = manual.copy()
manual_corr.columns = [
    int(c) if str(c).isdigit() else c
    for c in manual_corr.columns
]

unseen_manual_corr = unseen_manual.copy()
unseen_manual_corr.columns = [
    int(c) if str(c).isdigit() else c
    for c in unseen_manual_corr.columns
]

training_activity_smooth = ethovision_smooth.copy()
training_activity_smooth.columns = [
    int(c) if str(c).isdigit() else c
    for c in training_activity_smooth.columns
]

test_activity_smooth = unseen_ethovision_smooth.copy()
test_activity_smooth.columns = [
    int(c) if str(c).isdigit() else c
    for c in test_activity_smooth.columns
]

In [ ]:
training_correlation_df = calculate_video_immobility_times(
    manual_df=manual_corr,
    activity_df=training_activity_smooth,
    strain_by_exp=strain_by_exp,
    thresholds_by_strain=fixed_thresholds_by_strain,
    dataset_name="Training",
    fps=FPS,
    manual_interval_s=MANUAL_INTERVAL_S,
)

test_correlation_df = calculate_video_immobility_times(
    manual_df=unseen_manual_corr,
    activity_df=test_activity_smooth,
    strain_by_exp=unseen_strain_by_exp,
    thresholds_by_strain=fixed_thresholds_by_strain,
    dataset_name="Test",
    fps=FPS,
    manual_interval_s=MANUAL_INTERVAL_S,
)

In [ ]:
# ============================================================
# Manual vs automated immobility using the SAME trusted windows
# as the F1 analysis
# ============================================================

import numpy as np
import pandas as pd


def calculate_trusted_window_immobility(
    manual_df,
    activity_df,
    strain_by_exp,
    thresholds_by_strain,
    dataset_name,
    fps=25,
    interval_s=5,
    trust_s=2,
    align="center",
):
    """
    Calculates manual and automated immobility only within the same
    trusted windows used for the F1 analysis.

    Manual:
        each valid 'i' call contributes trust_s seconds

    Automated:
        frames within that call's trusted window are thresholded,
        then immobile frames are summed across all windows
    """

    rows = []

    for exp, strain in strain_by_exp.items():

        if exp not in manual_df.columns:
            print(f"Skipping Exp{exp} ({dataset_name}): not in manual dataframe")
            continue

        if exp not in activity_df.columns:
            print(f"Skipping Exp{exp} ({dataset_name}): not in automated dataframe")
            continue

        if strain not in thresholds_by_strain:
            print(f"Skipping Exp{exp}: no threshold for {strain}")
            continue

        manual_series = manual_df[exp]
        activity_series = pd.to_numeric(
            activity_df[exp],
            errors="coerce",
        )

        frame_idx, y_true = get_ground_truth_frames_aligned(
            manual_series=manual_series,
            activity_len=len(activity_series),
            fps=fps,
            interval_s=interval_s,
            trust_s=trust_s,
            align=align,
        )

        if len(frame_idx) == 0:
            print(f"Skipping Exp{exp}: no valid trusted windows")
            continue

        activity_values = activity_series.iloc[frame_idx].to_numpy()

        keep = ~np.isnan(activity_values)

        activity_values = activity_values[keep]
        y_true = y_true[keep]

        if len(activity_values) == 0:
            print(f"Skipping Exp{exp}: no valid automated values")
            continue

        threshold = thresholds_by_strain[strain]

        # Same coding as F1 analysis:
        # 0 = immobile, 1 = mobile
        y_pred = (activity_values > threshold).astype(int)

        manual_immobile_frames = np.sum(y_true == 0)
        automated_immobile_frames = np.sum(y_pred == 0)

        total_compared_frames = len(y_true)

        rows.append({
            "experiment": exp,
            "dataset": dataset_name,
            "strain": strain,

            "threshold": threshold,

            "n_compared_frames": total_compared_frames,
            "compared_duration_s": total_compared_frames / fps,

            "manual_immobility_s": manual_immobile_frames / fps,
            "automated_immobility_s": automated_immobile_frames / fps,

            "manual_immobility_percent":
                100 * manual_immobile_frames / total_compared_frames,

            "automated_immobility_percent":
                100 * automated_immobile_frames / total_compared_frames,
        })

    return pd.DataFrame(rows)


# Training set
training_correlation_df = calculate_trusted_window_immobility(
    manual_df=manual,
    activity_df=ethovision_smooth,
    strain_by_exp=strain_by_exp,
    thresholds_by_strain=fixed_thresholds_by_strain,
    dataset_name="Training",
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
)


# Test set
test_correlation_df = calculate_trusted_window_immobility(
    manual_df=unseen_manual,
    activity_df=unseen_ethovision_smooth,
    strain_by_exp=unseen_strain_by_exp,
    thresholds_by_strain=fixed_thresholds_by_strain,
    dataset_name="Test",
    fps=FPS,
    interval_s=WINDOW_S,
    trust_s=TRUST_S,
    align="center",
)


display(training_correlation_df)
display(test_correlation_df)

In [ ]:
# ============================================================
# Plot whole-video manual vs automated immobility
# One point = one animal/video
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, linregress
from matplotlib.lines import Line2D


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

correlation_df = test_correlation_df.copy()

# Combine the two MRL threshold groups for display only
correlation_df["strain_plot"] = correlation_df["strain"].replace({
    "MRL/MpJ_2": "MRL/MpJ"
})

# Remove incomplete observations
plot_df = (
    correlation_df
    .dropna(
        subset=[
            "manual_immobility_s",
            "automated_immobility_s",
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# Pearson correlation and linear regression
# ------------------------------------------------------------

x = plot_df["manual_immobility_s"].to_numpy(dtype=float)
y = plot_df["automated_immobility_s"].to_numpy(dtype=float)

pearson_r, pearson_p = pearsonr(x, y)
regression = linregress(x, y)

print(f"n = {len(plot_df)}")
print(f"Pearson r = {pearson_r:.4f}")
print(f"Pearson p = {pearson_p:.6g}")
print(f"R² = {regression.rvalue ** 2:.4f}")

display(
    plot_df[
        [
            "experiment",
            "strain_plot",
            "manual_immobility_s",
            "automated_immobility_s",
        ]
    ].sort_values(["strain_plot", "experiment"])
)


# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

strain_order = [
    "MRL/MpJ",
    "DBA/2J",
    "CBA/J",
    "C57BL/6J",
]

strain_colors = {
    "MRL/MpJ": "#82B8B8",
    "DBA/2J": "#E9A88F",
    "CBA/J": "#A98FC1",
    "C57BL/6J": "#91BD78",
}

# Distinct marker shapes so strains remain identifiable
# in grayscale / with achromatopsia
strain_markers = {
    "MRL/MpJ": "o",
    "DBA/2J": "s",
    "CBA/J": "^",
    "C57BL/6J": "D",
}


# ------------------------------------------------------------
# Create plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.3, 5.0))

for strain in strain_order:

    subset = plot_df[
        plot_df["strain_plot"] == strain
    ]

    if subset.empty:
        continue

    ax.scatter(
        subset["manual_immobility_s"],
        subset["automated_immobility_s"],
        s=62,
        marker=strain_markers[strain],
        facecolor=strain_colors[strain],
        edgecolor="#4A4A4A",
        linewidth=0.8,
        alpha=0.95,
        zorder=3,
    )


# ------------------------------------------------------------
# Axis limits
# ------------------------------------------------------------

data_max = max(x.max(), y.max())
data_min = min(x.min(), y.min())

padding = max(10, 0.06 * (data_max - data_min))

axis_min = max(0, data_min - padding)
axis_max = data_max + padding


# ------------------------------------------------------------
# Regression line
# ------------------------------------------------------------

line_x = np.linspace(axis_min, axis_max, 200)
line_y = regression.intercept + regression.slope * line_x

ax.plot(
    line_x,
    line_y,
    color="#333333",
    linewidth=1.7,
    zorder=2,
)


# ------------------------------------------------------------
# Optional identity line: automated = manual
# ------------------------------------------------------------

# ax.plot(
#     [axis_min, axis_max],
#     [axis_min, axis_max],
#     linestyle="--",
#     color="#A0A0A0",
#     linewidth=1.2,
#     zorder=1,
# )


# ------------------------------------------------------------
# Correlation annotation
# ------------------------------------------------------------

p_label = f"p = {pearson_p:.2e}"

# Uncomment if you want the statistics inside the panel
#
# ax.text(
#     0.05,
#     0.95,
#     (
#         f"Pearson r = {pearson_r:.2f}\n"
#         f"{p_label}\n"
#         f"n = {len(plot_df)}"
#     ),
#     transform=ax.transAxes,
#     ha="left",
#     va="top",
#     fontsize=13,
#     fontweight="bold",
#     color="#333333",
# )


# ------------------------------------------------------------
# Labels and styling
# ------------------------------------------------------------

ax.set_xlabel(
    "Manually scored immobility (s)",
    fontsize=14,
)

ax.set_ylabel(
    "Automated immobility (s)",
    fontsize=14,
)

ax.set_xlim(axis_min, axis_max)
ax.set_ylim(axis_min, axis_max)

ax.set_aspect("equal", adjustable="box")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#B5B5B5")
ax.spines["bottom"].set_color("#B5B5B5")

ax.tick_params(
    axis="both",
    labelsize=12,
    width=0.8,
)

ax.grid(False)


# ------------------------------------------------------------
# Strain legend with sample size
# ------------------------------------------------------------

strain_counts = (
    plot_df
    .groupby("strain_plot", observed=False)
    .size()
    .to_dict()
)

strain_handles = [
    Line2D(
        [0],
        [0],
        marker=strain_markers[strain],
        linestyle="none",
        markerfacecolor=strain_colors[strain],
        markeredgecolor="#4A4A4A",
        markeredgewidth=0.8,
        markersize=7,
        label=f"{strain} (n = {strain_counts[strain]})",
    )
    for strain in strain_order
    if strain in plot_df["strain_plot"].unique()
]



# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

plt.tight_layout()


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

plt.savefig(
    FIGURE_DIR / "manual_vs_automated_immobility_correlation.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight",
)

plt.savefig(
    FIGURE_DIR / "manual_vs_automated_immobility_correlation.pdf",
    format="pdf",
    dpi=600,
    bbox_inches="tight",
)
#plt.show()